# Download CelebA data

Pobiera obrazy z **oficjalnego** podziału train/test CelebA (`flwrlabs/celeba`),  
zbalansowane względem wybranego konceptu.

| split | źródło HF | liczba obrazów |
|-------|-----------|----------------|
| train | `split="train"` | `7 × DS_SIZE` (po połowie pos/neg) |
| test  | `split="test"`  | `1 × DS_SIZE` (po połowie pos/neg) |

Obrazy lądują w `data/images_{BALANCE_CONCEPT}`, metadane w `data/metadata_{BALANCE_CONCEPT}.csv`.

In [1]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

In [2]:
DS_SIZE = 256

# Koncept do balansowania — zmień na dowolny klucz z CONCEPTS poniżej
BALANCE_CONCEPT = 'straight_hair'


# HuggingFace attribute name -> nazwa kolumny w metadata
CONCEPTS = {
    'Eyeglasses':    'eyeglasses',
    'Bald':          'bald',
    'Big_Nose':      'big_nose',
    'Blond_Hair':    'blond_hair',
    'Black_Hair':    'black_hair',
    'Chubby':        'chubby',
    'Wearing_Hat':   'wearing_hat',
    'Gray_Hair':     'gray_hair',
    'Male':          'male',
    'Smiling':       'smiling',
    'Straight_Hair': 'straight_hair',
    'Wavy_Hair':     'wavy_hair',
    'Young':         'young',
}
# odwrócone mapowanie: kolumna -> atrybut HF
_COL_TO_HF = {v: k for k, v in CONCEPTS.items()}

assert BALANCE_CONCEPT in CONCEPTS.values(), (
    f"'{BALANCE_CONCEPT}' nie istnieje. Dostępne: {list(CONCEPTS.values())}"
)

ROOT = Path('..').resolve()
IMAGES_DIR   = ROOT / 'data' / f'images_{BALANCE_CONCEPT}'
METADATA_PATH = ROOT / 'data' / f'metadata_{BALANCE_CONCEPT}.csv'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

BALANCE_HF_ATTR = _COL_TO_HF[BALANCE_CONCEPT]

TRAIN_N = 7 * DS_SIZE   # 1792 -> po 896 pos/neg
TEST_N  = 1 * DS_SIZE   #  256 -> po 128 pos/neg

print(f'Koncept balansowania : {BALANCE_CONCEPT} (atrybut HF: {BALANCE_HF_ATTR})')
print(f'Train: {TRAIN_N} | Test: {TEST_N}')
print(f'Folder obrazów       : {IMAGES_DIR}')
print(f'Metadane             : {METADATA_PATH}')

Koncept balansowania : straight_hair (atrybut HF: Straight_Hair)
Train: 1792 | Test: 256
Folder obrazów       : E:\Do gita\WB2\data\images_straight_hair
Metadane             : E:\Do gita\WB2\data\metadata_straight_hair.csv


In [3]:
def collect_split(
    hf_split: str,
    n: int,
    split_label: str,
    start_idx: int,
    metadata_list: list,
) -> int:
    """
    Pobiera n obrazów z oficjalnego splitu HF, zbalansowanych względem BALANCE_HF_ATTR
    (n/2 pozytywnych, n/2 negatywnych). Zapisuje do IMAGES_DIR.
    """
    ds = load_dataset('flwrlabs/celeba', split=hf_split, streaming=True, token=HF_TOKEN)
    target_per_class = n // 2
    pos_count = neg_count = 0
    idx = start_idx

    print(f'\n[{split_label}] HF split="{hf_split}" | cel: {target_per_class} pos + {target_per_class} neg')
    for item in tqdm(ds):
        is_pos = bool(item[BALANCE_HF_ATTR])

        if is_pos and pos_count >= target_per_class:
            continue
        if not is_pos and neg_count >= target_per_class:
            continue

        img = item['image'].convert('RGB').resize((224, 224))
        filename = f'img_{idx:06d}.jpg'
        img.save(IMAGES_DIR / filename, quality=90)

        row = {'filename': filename, 'split': split_label}
        for hf_attr, col in CONCEPTS.items():
            row[col] = int(item[hf_attr])
        metadata_list.append(row)

        if is_pos:
            pos_count += 1
        else:
            neg_count += 1
        idx += 1

        if pos_count >= target_per_class and neg_count >= target_per_class:
            break

    print(f'  -> zebrano {pos_count} pos + {neg_count} neg = {idx - start_idx} obrazów')
    return idx

In [4]:
if METADATA_PATH.exists():
    print(f'Plik {METADATA_PATH} już istnieje.')
    print('Usuń go ręcznie jeśli chcesz pobrać dane od nowa.')
else:
    metadata_list = []
    idx = 0
    idx = collect_split('train', TRAIN_N, 'train', idx, metadata_list)
    idx = collect_split('test',  TEST_N,  'test',  idx, metadata_list)

    df = pd.DataFrame(metadata_list)
    col_order = ['filename', 'split'] + list(CONCEPTS.values())
    df = df[col_order]
    df.to_csv(METADATA_PATH, index=False)
    print(f'\nZapisano {len(df)} rekordów -> {METADATA_PATH}')

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]


[train] HF split="train" | cel: 896 pos + 896 neg


2754it [00:25, 108.54it/s]


  -> zebrano 896 pos + 896 neg = 1792 obrazów


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]


[test] HF split="test" | cel: 128 pos + 128 neg


612it [00:10, 60.13it/s] 

  -> zebrano 128 pos + 128 neg = 256 obrazów

Zapisano 2048 rekordów -> E:\Do gita\WB2\data\metadata_straight_hair.csv


## Podsumowanie pobranych danych

In [5]:
df = pd.read_csv(METADATA_PATH)
print(df.shape)
df.head()

(2048, 15)


,filename,split,eyeglasses,bald,big_nose,blond_hair,black_hair,chubby,wearing_hat,gray_hair,male,smiling,straight_hair,wavy_hair,young
0,img_000000.jpg,train,0,0,0,1,0,0,0,0,1,1,0,0,0
1,img_000001.jpg,train,0,0,1,0,0,0,0,0,1,1,0,0,0
2,img_000002.jpg,train,0,0,0,0,0,0,1,0,1,1,0,0,1
3,img_000003.jpg,train,0,0,1,1,0,0,0,0,1,1,1,0,1
4,img_000004.jpg,train,0,0,0,0,0,0,0,0,1,0,1,0,1


In [6]:
# Odsetek pozytywnych przypadków każdego konceptu per split
concept_cols = list(CONCEPTS.values())
summary = (
    df.groupby('split')[concept_cols]
    .mean()
    .mul(100)
    .round(1)
)
summary.columns.name = 'concept (% pozytywnych)'
summary

concept (% pozytywnych),eyeglasses,bald,big_nose,blond_hair,black_hair,chubby,wearing_hat,gray_hair,male,smiling,straight_hair,wavy_hair,young
split,,,,,,,,,,,,,
test,2.0,0.0,28.1,26.2,17.6,2.0,1.2,0.4,31.2,52.7,50.0,19.1,77.3
train,6.9,0.4,20.6,15.2,32.1,6.4,5.3,2.2,55.4,44.6,50.0,13.6,78.1
